<a href="https://colab.research.google.com/github/tharujayasinghe163/Statistical-Learning-e22163/blob/main/GPR_LR_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gaussian Process Regression

## Task: Explore modeling 'heating load' ($Y_1$) and 'cooling load' ($Y_2$) as a single-parameter Gaussian process.

### 1. Architectural Feasibility Analysis

A standard, baseline Gaussian Process Regression (GPR) model is fundamentally a scalar-valued simulator. It assumes a mapping from an input vector $\mathbf{x} \in \mathbb{R}^d$ to a single real-valued scalar output $y \in \mathbb{R}$, fully defined by a mean function $m(\mathbf{x})$ and a covariance kernel function $k(\mathbf{x}, \mathbf{x}')$:

$$f(\mathbf{x}) \sim \mathcal{GP}\left(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}')\right)$$

Because the [Energy Efficiency Dataset](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) requires the simultaneous prediction of **two continuous target variables**—Heating Load ($Y_1$) and Cooling Load ($Y_2$)—a single standard, unmodified "single-parameter" GP cannot natively output both variables at once as a joint system.

To overcome this limitation within a Gaussian Process framework, we must evaluate three potential design strategies:

---

### 2. Evaluated Modeling Strategies

#### Strategy A: Independent Single-Output GPs (The Baseline)

We can train two entirely separate, decoupled Gaussian Processes:

* $\mathcal{GP}_1: \mathbf{x} \to Y_1$ (Heating Load)
* $\mathcal{GP}_2: \mathbf{x} \to Y_2$ (Cooling Load)
* **Critique:** While trivial to implement using standard toolkits like `scikit-learn`'s `GaussianProcessRegressor`, this approach operates under the flawed assumption that $Y_1$ and $Y_2$ are conditionally independent given the inputs. In building thermodynamics, heating and cooling loads are heavily governed by the same underlying physical properties (e.g., overall insulation, surface area, and glazing), meaning their errors and uncertainties are highly correlated. Discarding this cross-target correlation leads to sub-optimal predictive uncertainty bounds.

#### Strategy B: Multi-Task Gaussian Process (Intrinsic Coregionalization)

To genuinely model both outputs within a unified framework, we can implement a **Multi-Task GP** using an Intrinsic Coregionalization Model (ICM) kernel. This expands the covariance structure into a Kronecker product:

$$K((\mathbf{x}, i), (\mathbf{x}', j)) = k_{\text{inputs}}(\mathbf{x}, \mathbf{x}') \otimes B_{ij}$$

Where:

* $k_{\text{inputs}}(\mathbf{x}, \mathbf{x}')$ is a standard spatial kernel (such as a Matérn or RBF kernel) evaluating building feature similarities.
* $B$ is a $2 \times 2$ positive semi-definite matrix that explicitly parameterizes the task-to-task correlation between Heating and Cooling loads.
* **Critique:** This represents the most statistically rigorous method. It allows the model to share data structure across both tasks, leading to more accurate predictions and tighter, more reliable confidence intervals—especially in scenarios where data points for one of the loads might be missing or noisy.

#### Strategy C: Single-GP Data Augmentation (The Structural Trick)

If we are strictly constrained to utilizing a *single standard single-parameter GP instance*, we can restructure the dataset via data augmentation:

1. Duplicate the input feature matrix $\mathbf{X}$ to create a matrix of size $2N \times 9$.
2. Append a 9th indicator feature ($X_9$): set $X_9 = 0$ for rows corresponding to Heating Load, and $X_9 = 1$ for Cooling Load.
3. Flatten the target matrices into a single long vector $\mathbf{Y} \in \mathbb{R}^{2N}$.

* **Critique:** This transforms the multi-output problem into a single-output problem, fulfilling the literal constraint of using a single GP parameter framework. However, capturing the proper interaction between the categorical task indicator ($X_9$) and the structural building features requires meticulous kernel design (e.g., using product or change-of-input kernels).

---

### 3. Final Conclusions & Engineering Recommendations

* **Direct Answer:** No, it is not possible to model both heating and cooling loads as a single-parameter, single-output Gaussian Process using a naive, out-of-the-box configuration.
* **Physical Domain Alignment:** From a thermodynamic perspective, heating and cooling loads are dual expressions of a building envelope's efficiency. Because they share a high statistical and physical correlation, modeling them independently wastes valuable mutual information.
* **Final Recommendation:** The ideal modeling selection for this dataset is a **Multi-Task Gaussian Process (Strategy B)** implemented via modern frameworks like `GPyTorch` or `GPflow`. If restricted purely to simple single-output architectures, **Strategy C (Data Augmentation)** provides a valid mathematical workaround, but standard independent tracking (**Strategy A**) should be avoided if robust uncertainty quantification is required.

Linear Regression

## Task: Explore predicting 'predicted_energy_demand' using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

### 1. Theoretical Framework & Mathematical Formulation

To model the continuous target variable **`predicted_energy_demand`** ($Y$), we formulate a Multiple Linear Regression (MLR) model. The model assumes a linear combination of selected building and environmental predictors ($X_i$) along with an additive error term $\epsilon$:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p + \epsilon$$

Where:

* $\beta_0$ is the intercept.
* $\beta_i$ represent the ordinary least squares (OLS) regression coefficients for each selected predictor.
* $\epsilon \sim \mathcal{N}(0, \sigma^2)$ represents independent, identically distributed Gaussian residual errors.

---

### 2. Feature Selection & Justification of Parameters

Based on the physics of green building design and the metadata provided in the [Green Building Multi-Source Environment Dataset](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset), variables are categorized and selected using thermodynamic principles:

| Parameter Category | Chosen Predictors | Engineering & Thermodynamic Justification |
| --- | --- | --- |
| **Direct Energy Drivers** | `heating_energy`, `cooling_energy`, `electricity_consumption` | **Highly Critical:** These operational metrics have a direct, additive accounting relationship with total overall energy demand ($Y$). They are expected to hold strong positive linear coefficients. |
| **Human / Behavioral Factors** | `occupancy`, `activity_level` | **Critical:** Human presence introduces internal metabolic heat gains and dictates building management system (BMS) scheduling. Higher occupancy linearly scales ventilation and power consumption requirements. |
| **Environmental Boundary Conditions** | `outdoor_temperature`, `solar_radiation`, `ventilation_rate` | **Critical:** `outdoor_temperature` and `solar_radiation` govern the thermal gradient across the building envelope, directly forcing HVAC systems to work harder. `ventilation_rate` dictates how much unconditioned outside air must be mechanically treated. |
| **Excluded Parameters** | `indoor_temperature`, `indoor_humidity`, `indoor_lighting`, `indoor_noise` | **Omitted due to Multicollinearity:** Indoor environmental comfort indicators are *consequences* of energy consumption (e.g., active HVAC usage keeps indoor temperature stable), rather than independent drivers. Including them creates severe multicollinearity, inflating the variance of coefficient estimates. |

---

### 3. Expected Model Evaluation Metrics & Discussion

When evaluating the performance of the OLS linear regression model on this dataset, the following diagnostics must be analyzed:

#### A. Goodness-of-Fit ($R^2$ and Adjusted $R^2$)

* **Expectation:** The Coefficient of Determination ($R^2$) is expected to be exceptionally high ($\geq 0.90$).
* **Discussion:** Because `predicted_energy_demand` is a synthetic target derived structurally from operational loads (`heating_energy`, `cooling_energy`, and `electricity_consumption`), a linear configuration will successfully capture the vast majority of the target's variance. The Adjusted $R^2$ should be used to confirm that adding environmental parameters (like wind speed or rainfall) provides genuine explanatory power rather than overfitting noise.

#### B. Analysis of Coefficients (Significance & Interpretability)

* **Operational Weights:** The coefficients ($\beta_i$) for `heating_energy` and `cooling_energy` will be positive and highly statistically significant ($p < 0.001$).
* **Baseline Loads:** The intercept ($\beta_0$) represents the baseline background energy demand of the building when empty and unconditioned (e.g., static server loads, architectural safety lighting).

#### C. Verification of OLS Assumptions (Residual Diagnostics)

To validate the robustness of the linear model, the residuals must satisfy classical linear regression assumptions:

1. **Homoscedasticity:** A plot of residuals vs. predicted values should show a random dispersion of errors. If variance expands at higher demands, a log-transform on the target variable ($\log(Y)$) should be considered.
2. **Normality:** A Q-Q plot of the residuals should ideally fall along a straight $45^\circ$ line, validating the use of $t$-tests for coefficient significance.

---

### 4. Direct Conclusion

Predicting `predicted_energy_demand` using a multiple linear regression model is **highly feasible and highly effective** for this dataset. This effectiveness is due to the strong, physically deterministic linear relationships between active equipment energy consumption parameters and total cumulative building demand. To ensure optimal model generalizability, a feature set consisting of **direct operational loads, occupancy metrics, and core ambient weather drivers** should be selected, while redundant indoor comfort variables should be pruned to prevent variance inflation via multicollinearity.